# Análisis de experimentos federados con XGBoost (Adidas)


Este cuaderno reúne todo lo necesario para:
1. **Lanzar o relanzar experimentos** definidos en `configs/`.
2. **Cargar métricas** guardadas en `results/`.
3. **Graficar** evolución de RMSE, tiempos y **analizar la importancia de variables**.

In [10]:
#import os
#os.system("pip install -e .")



In [11]:
import pandas as pd
from pathlib import Path
from itertools import product
from pathlib import Path
import toml, datetime
import subprocess, time

## Generar configuraciones TOML

In [12]:
from itertools import product
from pathlib import Path
import datetime
import toml

CONFIG_DIR = Path("../configs")
CONFIG_DIR.mkdir(exist_ok=True)

# Grids ajustados
partitioners  = ["retailer_region"]
strategies    = ["bagging", "cycling"]
test_fracs    = [0.1, 0.2]           # solo dos fracciones de test
local_epochs  = [10, 15, 20, 25]           # pocos epochs locales para controlar ensemble
etas          = [0.002, 0.003, 0.004, 0.005]       # learning rates moderados y bajos
max_depths    = [4, 6]            # profundidades más contenidas
subsamples    = [0.6, 0.8]           # subsamples estándar
scaled_lr     = True                 # activa el scaled learning rate

generated = []
today = datetime.date.today().isoformat()

for part, strat, tf, le, eta, md, ss in product(
        partitioners, strategies,
        test_fracs, local_epochs, etas, max_depths, subsamples
):
    name = (f"{part}-{strat}-ce-on-tf-{tf}-le-{le}"
            f"-eta-{eta}-md-{md}-ss-{ss}").replace(".", "p")

    cfg = {
        "run-id": name + "_" + today,
        "strategy": strat,
        "partitioner": part,
        "centralised-eval": True,
        "test-fraction": tf,
        "local-epochs": le,
        "scaled-lr": scaled_lr,
        "params": {
            "eta": eta,
            "max_depth": md,
            "subsample": ss,
        },
    }

    toml.dump(cfg, open(CONFIG_DIR / f"{name}.toml", "w", encoding="utf-8"))
    generated.append(name)

print(f"📝 {len(generated)} configuraciones creadas en {CONFIG_DIR}")



📝 256 configuraciones creadas en ..\configs


## Ejecutar Batch de experimentos

### Un único experimento de prueba

In [13]:
import random, subprocess, time
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SRC_DIR      = PROJECT_ROOT / "xgboost_comprehensive"
CONFIG_DIR   = PROJECT_ROOT / "configs"

# Usamos la CLI flwr en vez de python -m flwr
FLWR_CMD     = ["flwr", "run", ".", "--run-config"]
K            = 5

all_cfgs    = sorted(CONFIG_DIR.glob("*.toml"))
random_cfgs = random.sample(all_cfgs, k=min(K, len(all_cfgs)))

for i, cfg in enumerate(random_cfgs, 1):
    print(f"({i}/{K}) 🚀  {cfg.name}")
    t0 = time.perf_counter()

    # Invocamos la CLI flwr que ya tienes en PATH
    proc = subprocess.Popen(
        FLWR_CMD + [str(cfg)],
        cwd=str(SRC_DIR.parent),      # cwd en src/ para que flwr importe tu paquete
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    # Streaming en vivo de stdout+stderr
    for line in proc.stdout:
        print(line, end="")

    proc.wait()
    dt = time.perf_counter() - t0
    print(f"\n⏱️  {dt:.1f}s — returncode: {proc.returncode}\n")
    if proc.returncode != 0:
        print("⚠️ ERROR detectado, revisa el log de arriba para ver la causa\n")





(1/5) 🚀  retailer_region-cycling-ce-on-tf-0p1-le-20-eta-0p002-md-6-ss-0p8.toml
Loading project configuration... 
Success
[DEBUG init_server_csv] path=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\bagging\server_metrics.csv metrics=['rmse', 'mae', 'r2', 'eval_time_round']
[DEBUG init_server_csv] path=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\cycling\server_metrics.csv metrics=['rmse', 'mae', 'r2', 'eval_time_round']
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 1 clients (out of 28)
(ClientAppActor 

### Conjunto de experimentos

In [ ]:
#!/usr/bin/env python3
import subprocess
import time
from pathlib import Path

# Ajusta estas rutas según tu estructura
PROJECT_ROOT = Path("..").resolve()
SRC_DIR      = PROJECT_ROOT / "xgboost_comprehensive"
CONFIG_DIR   = PROJECT_ROOT / "configs"

# Comando base de Flower CLI
FLWR_CMD = ["flwr", "run", ".", "--run-config"]

# Listamos y ordenamos todas las configuraciones .toml
all_cfgs = sorted(CONFIG_DIR.glob("*.toml"))

total = len(all_cfgs)
if total == 0:
    print("⚠️  No se han encontrado archivos .toml en", CONFIG_DIR)
    exit(1)

for idx, cfg in enumerate(all_cfgs, start=1):
    print(f"({idx}/{total}) 🚀  Lanzando experimento con {cfg.name}")
    t0 = time.perf_counter()

    # Iniciamos Flower desde la carpeta padre de src/ para que importe tu paquete
    proc = subprocess.Popen(
        FLWR_CMD + [str(cfg)],
        cwd=str(SRC_DIR.parent),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    # Stream de stdout+stderr en tiempo real
    for line in proc.stdout:
        print(line, end="")

    proc.wait()
    dt = time.perf_counter() - t0

    print(f"\n⏱️  Tiempo: {dt:.1f}s — returncode: {proc.returncode}\n")
    if proc.returncode != 0:
        print("⚠️ ERROR detectado. Revisa el log anterior para más detalles.\n")


(1/256) 🚀  Lanzando experimento con retailer_region-bagging-ce-on-tf-0p1-le-10-eta-0p002-md-4-ss-0p6.toml
Loading project configuration... 
Success
[DEBUG init_server_csv] path=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\bagging\server_metrics.csv metrics=['rmse', 'mae', 'r2', 'eval_time_round']
[DEBUG init_server_csv] path=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\cycling\server_metrics.csv metrics=['rmse', 'mae', 'r2', 'eval_time_round']
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
[DEBUG evaluate_fn] round=0, csv=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\resul

In [ ]:
#!/usr/bin/env python3
import subprocess
import time
from pathlib import Path

# ---------- PARAMETROS ----------
PROJECT_ROOT = Path("..").resolve()
SRC_DIR      = PROJECT_ROOT / "xgboost_comprehensive"
CONFIG_DIR   = PROJECT_ROOT / "configs"

FLWR_CMD     = ["flwr", "run", ".", "--run-config"]
START_FROM   = 196             # ← primera posición (1-based) desde la que reanudar
# --------------------------------

all_cfgs = sorted(CONFIG_DIR.glob("*.toml"))
total    = len(all_cfgs)

if total == 0:
    print("⚠️  No se han encontrado archivos .toml en", CONFIG_DIR)
    raise SystemExit(1)
if START_FROM > total:
    print(f"⚠️  START_FROM={START_FROM} supera el total de ficheros ({total})")
    raise SystemExit(1)

for idx, cfg in enumerate(all_cfgs[START_FROM-1:], start=START_FROM):
    print(f"({idx}/{total}) 🚀  Lanzando experimento con {cfg.name}")
    t0 = time.perf_counter()

    proc = subprocess.Popen(
        FLWR_CMD + [str(cfg)],
        cwd=str(SRC_DIR.parent),          # garantizamos import de tu paquete
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    # Stream de stdout+stderr en vivo
    for line in proc.stdout:
        print(line, end="")

    proc.wait()
    dt = time.perf_counter() - t0

    print(f"\n⏱️  Tiempo: {dt:.1f}s — returncode: {proc.returncode}\n")
    if proc.returncode != 0:
        print("⚠️  ERROR detectado. Detén la ejecución y revisa el log.\n")
        break           # evita seguir si algo falló


(196/256) 🚀  Lanzando experimento con retailer_region-cycling-ce-on-tf-0p2-le-10-eta-0p002-md-6-ss-0p8.toml
Loading project configuration... 
Success
[DEBUG init_server_csv] path=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\bagging\server_metrics.csv metrics=['rmse', 'mae', 'r2', 'eval_time_round']
[DEBUG init_server_csv] path=C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\cycling\server_metrics.csv metrics=['rmse', 'mae', 'r2', 'eval_time_round']
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 1 clients